In [1]:
import json 
import requests

In [2]:
from bs4 import BeautifulSoup 
import urllib3

In [3]:
MSAMB_URL = "https://www.msamb.com/ApmcDetail/APMCPriceInformation"

In [4]:
import pandas as pd 
import difflib 
import json 
import requests
import urllib3
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait, Select
from selenium.webdriver.support import expected_conditions as EC
import time
from io import StringIO


In [5]:
import selenium
selenium.__version__

'4.40.0'

In [9]:
def __fetch_table_from_url(url,selector, type):
    MSAMB_URL = url 
    brave_path = '/opt/brave.com/brave/brave'

    # 1. OPTIMIZE BROWSER OPTIONS FOR SPEED
    options = webdriver.ChromeOptions()
    options.binary_location = brave_path
    options.add_argument('--headless=new')           # Do not open a visible browser window
    options.add_argument('--disable-gpu')            # Disable GPU hardware acceleration
    options.add_argument('--no-sandbox')             # Bypass OS security model (required for headless on Linux)
    options.add_argument('--disable-dev-shm-usage')  # Overcome limited resource problems
    options.add_argument('--blink-settings=imagesEnabled=false') # Do not load images (Huge speed boost)
    options.page_load_strategy = 'eager'   
              # Don't wait for CSS/Images to fully load

    # Initialize WebDriver
    service = Service()
    driver = webdriver.Chrome(options=options)
    wait = WebDriverWait(driver, 10)

    try:
        print("🚀 Loading page...")
        driver.get(MSAMB_URL)
        
        # 2. HANDLE LANGUAGE SELECTION
        lang_element = wait.until(EC.element_to_be_clickable((By.ID, "language")))
        lang_select = Select(lang_element)
        
        if "English" not in lang_select.first_selected_option.text:
            print("🌐 Switching to English...")
            # Get a reference to the dropdown BEFORE changing language
            drp_commodities = driver.find_element(By.ID, "drpCommodities")
            lang_select.select_by_visible_text("English")
            
            # Wait until the old dropdown is destroyed (meaning the page reloaded/updated)
            wait.until(EC.staleness_of(drp_commodities))
            
            # Wait for the new dropdown to be clickable
            wait.until(EC.element_to_be_clickable((By.ID, "drpCommodities")))
            
        title_element = wait.until(EC.element_to_be_clickable((By.ID, "APMCTitle")))
        title_element.click()
        time.sleep(1)
        option_element = wait.until(EC.element_to_be_clickable((By.LINK_TEXT, type)))
        option_element.click()
        time.sleep(1)

        # 3. SELECT COMMODITY & WAIT FOR AJAX REFRESH
        print("🌾 Selecting value")
        if type == "Commodity- District Wise":
            drp_element = wait.until(EC.presence_of_element_located((By.ID, "drpCommodityDistrict")))
        else:
            drp_element = wait.until(EC.presence_of_element_located((By.ID, "drpDistrictCommodity")))
            
        dropdown = Select(drp_element)
            
        selector_name = selector
        found = False
        
        for opt in dropdown.options:
            if selector_name.upper() in opt.text.upper():
                # Grab a reference to the CURRENT table before we click
                
                dropdown.select_by_visible_text(opt.text)
                found = True
                time.sleep(1)
                # THE MAGIC WAIT: Wait until the old table disappears from the DOM 
                # This guarantees the AJAX request finished and the new data is rendering
                break

        if not found:
            print(f"❌ Selector '{selector_name}' not found in dropdown.")
        else:
            # 4. EXTRACT DATA
            print("📊 Extracting table...")
            # Wait for the NEW table to be present
            if type == "Commodity- District Wise":
                target_div_id = "CommodityDistrictGird"
            else:
                target_div_id = "DistrictCommodityGird"
                
            # 2. Build a CSS Selector that looks INSIDE that specific Div
            table_selector = f"div#{target_div_id} table.custom-table"
            
            # 3. Wait for and grab that exact table
            table_element = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, table_selector)))
            table_html = table_element.get_attribute('outerHTML')
            
            # 4. Convert to Pandas
            from io import StringIO
            df = pd.read_html(StringIO(table_html))[0]
            
            return df
            
            # print(df.head())
            
    except Exception as e:
        driver.save_screenshot("selenium_crash.png")
        raise e

    # finally:
        # Always close the browser, even if the code crashes, to prevent memory leaks
        driver.quit()


In [10]:
df = __fetch_table_from_url(MSAMB_URL, "onion", "Commodity- District Wise")

🚀 Loading page...
🌐 Switching to English...
🌾 Selecting value
📊 Extracting table...


In [11]:
df

,District,Variety,Unit,Quantity,Lrate,Hrate,Mrate
0,25/02/2026,25/02/2026,25/02/2026,25/02/2026,25/02/2026,25/02/2026,25/02/2026
1,AHILYANAGAR,LAL,QUINTAL,42805,263,1288,856
2,AHILYANAGAR,UNHALI,QUINTAL,8122,151,1500,826
3,AKOLA,----,QUINTAL,560,500,1400,1000
4,AMRAVATI,LAL,QUINTAL,407,600,2400,1500
...,...,...,...,...,...,...,...
278,SATARA,LOCAL,QUINTAL,45,1000,1300,1200
279,SOLAPUR,LAL,QUINTAL,37554,100,2000,800
280,THANE,No. 1,QUINTAL,3,1400,1500,1450
281,THANE,No. 2,QUINTAL,3,1200,1300,1250
